In [1]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    dataset.prepareDataset(
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
    shuffle=False
    )
    from qsprpred.data.descriptors.sets import RDKitDescs
    
    rdkit_descs = RDKitDescs()
    
    dataset.addDescriptors([rdkit_descs])
    
    dataset.descriptorSets
    return dataset
    

In [3]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding
from sklearn.base import BaseEstimator, TransformerMixin

class SMILESDataset(Dataset):
    def __init__(self, smiles, tokenizer, max_len=128):
        self.smiles = smiles
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        smile = self.smiles[idx]
        encoding = self.tokenizer(smile, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return encoding


class ChemBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name="entropy/roberta_zinc_480m", max_len=128, batch_size=32, device=None):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = RobertaForMaskedLM.from_pretrained(self.model_name).to(self.device)
        self.tokenizer = RobertaTokenizerFast.from_pretrained(self.model_name, max_len=self.max_len)
        self.collator = DataCollatorWithPadding(self.tokenizer, padding=True, return_tensors='pt')
        self.embedding_dim = None  # bude nastaven po fit()

    def fit(self, X, y=None):
        # Zjistíme embedding dimenzi na prvním SMILES
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=1, collate_fn=self.collator)
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                embedding = outputs[1][-1]  # poslední hidden state
                self.embedding_dim = embedding.shape[-1]
                break
        return self

    def transform(self, X):
        self.model.eval()
        smiles_dataset = SMILESDataset(X, self.tokenizer, max_len=self.max_len)
        dataloader = DataLoader(smiles_dataset, batch_size=self.batch_size, collate_fn=self.collator)
        embeddings_list = []

        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch['input_ids'].squeeze(1).to(self.device)
                attention_mask = batch['attention_mask'].squeeze(1).to(self.device)
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
                full_embeddings = outputs[1][-1]
                embeddings = ((full_embeddings * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(-1).unsqueeze(-1))
                embeddings_list.append(embeddings)

        all_embeddings = torch.cat(embeddings_list, dim=0).cpu().numpy()
        column_names = [f"chemberta_{i}" for i in range(self.embedding_dim)]
        return pd.DataFrame(all_embeddings, columns=column_names)


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("CK1/data/ck1_train_1")

X2_all = load_datasets("CK1/data/ck1_val_1")

X3_all = load_datasets("CK1/data/ck1_test_1")

transformer = ChemBERTaTransformer()
X_train_emb = transformer.fit_transform(X1_all.df["Drug"])
X_val_emb = transformer.transform(X2_all.df["Drug"])
X_test_emb = transformer.transform(X3_all.df["Drug"])


/tmp/ipykernel_28924/195630644.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smile = self.smiles[idx]


In [5]:
X1_all.df.Y.sum()/X1_all.df.Y.count()

0.1864754098360656

In [6]:
import pandas as pd
X1_all.X = X1_all.X.reset_index(drop=True)
X_train_emb = X_train_emb.reset_index(drop=True)
X1_all.X = pd.concat([X1_all.X, X_train_emb], axis = 1)

In [7]:
X2_all.X = X2_all.X.reset_index(drop=True)
X_val_emb = X_val_emb.reset_index(drop=True)
X2_all.X = pd.concat([X2_all.X, X_val_emb], axis = 1)
X3_all.X = X3_all.X.reset_index(drop=True)
X_test_emb = X_test_emb.reset_index(drop=True)
X3_all.X = pd.concat([X3_all.X, X_test_emb], axis = 1)

In [8]:
X1 = X1_all.X
y1 = X1_all.y
X2 = X2_all.X
y2 = X2_all.y
X3 = X3_all.X
y3 = X3_all.y

In [9]:
imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [10]:
pd.DataFrame(X1).columns[pd.DataFrame(X1).isna().any()].tolist()



[]

In [11]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001
0,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.867205,1.562896,0.740014,-0.007127,0.302924,0.290818,-1.193038,0.389139,-0.488946,0.195621
1,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.010128,0.448419,-1.361342,0.603048,-0.248246,0.183608,-0.284349,-1.654112,0.848054,-0.329638
2,-0.090909,2.722828,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.617997,-0.223711,1.272375,-0.984488,0.465848,0.762726,1.492865,0.138268,1.119030,-0.139090
3,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-1.339097,0.116163,0.138211,1.862946,-1.349124,0.570259,0.866984,0.076977,-1.038055,-0.966065
4,-0.090909,-0.367265,-0.090909,4.053217,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,0.004662,-0.724672,-0.511796,0.711238,2.620635,-0.227298,0.492240,2.185143,-1.080375,0.130744
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,-0.090909,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,0.718201,-1.459093,-0.386884,-1.513505,-1.252675,-2.065734,1.243633,0.522379,0.473159,0.724154
591,3.466833,-0.367265,-0.090909,-0.246718,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,0.801147,-1.028012,-0.336019,-1.439344,-1.121564,-1.984827,1.271151,0.336867,0.342814,0.574947
592,-0.090909,0.209094,-0.090909,0.555300,0.0,-0.137074,0.0,-0.090909,-0.251358,-0.111571,...,-0.263113,0.870566,0.721030,-0.434981,-0.480444,0.155046,0.349937,-1.240764,-0.779170,0.998289
593,-0.090909,-0.367265,-0.090909,-0.246718,0.0,6.992298,0.0,-0.090909,-0.251358,-0.111571,...,-1.788569,-0.132144,-0.171981,-1.256847,-0.091291,0.619601,-1.287513,0.745633,-0.119098,0.476874


In [12]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [13]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, matthews_corrcoef
import pandas as pd

def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    val_mcc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        val_mcc_t.append(matthews_corrcoef(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
        print(matthews_corrcoef(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    my_df["MCC"] = val_mcc_t
    return my_df

In [14]:
import optuna
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
import torch
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import confusion_matrix
def objective(trial, X_train, y_train, X_test, y_test):
    dropout_frac = trial.suggest_categorical("dropout_frac", [0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.8, 0.9])
    patience = trial.suggest_categorical("patience", [10, 40, 75])
    tol = trial.suggest_categorical("tol", [1e-5, 1e-4, 1e-3, 1e-2, 0])
    weight_decay = trial.suggest_categorical("weight_decay", [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 0])
    n_epochs = trial.suggest_categorical("n_epochs", [200, 300, 500, 1000])
    batch_size = trial.suggest_categorical("batch_size", [1024, 512, 256, 128, 64])
    optimizer = trial.suggest_categorical("optimizer", ["optim.AdamW", "optim.RMSprop"])
    lr = trial.suggest_categorical("lr", [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
    neuron_layers_dict = {
    '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[2048, 1024, 512, 256, 128, 64, 32, 16, 8]': [2048, 1024, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 1024, 256, 64, 8]': [4096, 1024, 256, 64, 8],
    '[4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8]': [4096, 3072, 2048, 1536, 1024, 768, 512, 256, 128, 64, 32, 16, 8],
    '[4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096]': [4096, 2048, 1024, 512, 256, 512, 1024, 2048, 4096],
    '[200]': [200],
    '[2000]': [2000],
    '[2000, 1000]': [2000, 1000],
    '[2000, 1000, 500]': [2000, 1000, 500],
    '[1000, 50]': [1000, 50],
    '[4000, 2000]': [4000, 2000],
    '[4000, 2000, 1000, 500]': [4000, 2000, 1000, 500],
    '[4000, 2000, 2000, 500]': [4000, 2000, 2000, 500]
    }
    neuron_layers_size = trial.suggest_categorical("neuron_layers_size", list(neuron_layers_dict.keys()))
    opt = {"optim.AdamW": optim.AdamW,
           "optim.RMSprop": optim.RMSprop}
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)
    
    # Model
    model = STFullyConnected(
        n_dim=X_train.shape[1],
        n_class=1,
        gpus=[],
        device=device,
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=dropout_frac,
        patience=patience,
        tol=tol,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=weight_decay,
        n_epochs=n_epochs,
        neuron_layers= neuron_layers_dict[neuron_layers_size],  # Použití neuron_layers_size
        batch_size=batch_size,
        optimizer=opt[optimizer],
        lr=lr,
        random_seed=69
    )
    # Trénink a predikce
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds_bin = preds > 0.5

    # Metiky
    f1 = f1_score(y_test, preds_bin)
    acc = accuracy_score(y_test, preds_bin)
    mcc = matthews_corrcoef(y_test, preds_bin)
    
    # Můžeš logovat i do trialu
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("acc", acc)
    display(confusion_matrix(y_test, preds_bin))
    return mcc  # maximalizujeme MCC


In [ ]:
study_3 = optuna.create_study(
    study_name="CK1_study_bert",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.NSGAIISampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=200
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

[I 2025-04-29 22:34:56,559] Using an existing study with name 'CK1_study_bert' instead of creating a new one.


cuda


array([[117,   8],
       [ 14,  16]])

[I 2025-04-29 22:35:11,938] Trial 1804 finished with value: 0.5125730465744756 and parameters: {'dropout_frac': 0.1, 'patience': 10, 'tol': 0.001, 'weight_decay': 0.001, 'n_epochs': 200, 'batch_size': 1024, 'optimizer': 'optim.AdamW', 'lr': 1e-06, 'neuron_layers_size': '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]'}. Best is trial 918 with value: 0.5125730465744756.


cuda


...

[I 2025-04-29 23:30:20,171] Trial 1981 finished with value: 0.46266666666666667 and parameters: {'dropout_frac': 0.1, 'patience': 10, 'tol': 0.01, 'weight_decay': 0.0001, 'n_epochs': 200, 'batch_size': 256, 'optimizer': 'optim.AdamW', 'lr': 1e-06, 'neuron_layers_size': '[4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]'}. Best is trial 918 with value: 0.5125730465744756.


cuda


array([[89, 36],
       [11, 19]])

[I 2025-04-29 23:30:25,947] Trial 1982 finished with value: 0.2851496151677274 and parameters: {'dropout_frac': 0.1, 'patience': 10, 'tol': 0.0001, 'weight_decay': 0.0001, 'n_epochs': 200, 'batch_size': 1024, 'optimizer': 'optim.AdamW', 'lr': 1e-06, 'neuron_layers_size': '[2000]'}. Best is trial 918 with value: 0.5125730465744756.


cuda


In [ ]:
study_3 = optuna.create_study(
    study_name="CK1_study_bert",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=500
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

In [ ]:
print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"))


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())


In [ ]:
study_3 = optuna.create_study(
    study_name="CK1_study_bert",  # jméno pro pozdější načtení
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True  # pokud už existuje, nepřepíše ji
)

# Spusť optimalizaci
study_3.optimize(
    lambda trial: objective(trial, X1, y1, X2, y2),
    n_trials=1000
)
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

# Pokud chceš F1 a ACC u nejlepšího modelu:
print("Best F1:", study_3.best_trial.user_attrs["f1"])
print("Best ACC:", study_3.best_trial.user_attrs["acc"])

In [ ]:
model2 = STFullyConnected(
        n_dim=X1.shape[1],
        n_class=1,
        gpus=[],
        device="cuda",
        is_reg=False,
        act_fun=F.selu,
        random_seed=69,
    **study_3.best_params
    )

In [ ]:
display(study_3.best_params)

In [ ]:
lol = study_3.best_params.copy()

In [ ]:
lol

In [ ]:
print("Best MCC:", study_3.best_value)
print("Best parameters:", study_3.best_params)

In [ ]:
model_2 = STFullyConnected(
        n_dim=X1.shape[1],
        n_class=1,
        gpus=[],
        device="cuda",
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=0.1,
        patience=10,
        tol=0.001,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=0.0001,
        n_epochs=300,
        neuron_layers= [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],  # Použití neuron_layers_size
        batch_size=1024,
        optimizer=optim.AdamW,
        lr=1e-6,
        random_seed=69
    )
model_2.fit(X1, y1, X2, y2)

In [ ]:
pred_val = model_2.predict(X3)

In [ ]:
pred_val = pred_val > 0.5

In [ ]:
from sklearn.metrics import matthews_corrcoef
print(matthews_corrcoef(pred_val, y3))

In [ ]:
import numpy as np
import pandas as pd

# Převod na DataFrame
X_final_train = pd.concat([pd.DataFrame(X1), pd.DataFrame(X2)], axis=0)
y_final_train = pd.concat([pd.DataFrame(y1), pd.DataFrame(y2)], axis=0)

# Pokud chceš mít je jako DataFrame s jedním sloupcem pro y


print(type(X_final_train))
print(type(y_final_train))

In [ ]:
model_final = STFullyConnected(
        n_dim=X_final_train.shape[1],
        n_class=1,
        gpus=[],
        device="cuda",
        is_reg=False,
        act_fun=F.selu,
        dropout_frac=0.1,
        patience=10,
        tol=0.001,  # Opraveno: nyní používáme hodnotu z trial
        weight_decay=0.0001,
        n_epochs=252,
        neuron_layers= [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8],  # Použití neuron_layers_size
        batch_size=1024,
        optimizer=optim.AdamW,
        lr=1e-6,
        random_seed=69
    )
model_final.fit(X_final_train, y_final_train)

In [ ]:
pred_test = model_final.predict(X2)
pred_test = pred_test > 0.5
print(matthews_corrcoef(pred_test, y2))